# FloorPlanCAD Object Detection — Colab Training
**Trước khi chạy**: `Runtime` → `Change runtime type` → **GPU (T4 hoặc A100)**

### Cấu trúc storage
```
/content/              ← Colab SSD (nhanh, tạm thời)
  object_detection/    ← code (git clone mỗi session)
  FloorPlanCAD_orig/   ← raw data (unzip từ Drive)
  FloorPlanCAD_data/   ← processed dataset

Drive/MyDrive/Research/multi_model/object_detection/
  data/zips/           ← zip gốc (download 1 lần)
  data/dataset.tar.gz  ← processed dataset (build 1 lần)
  checkpoints/         ← model weights
```


## 1 — Setup: GPU + Drive + Clone


In [ ]:
# Kiểm tra GPU
import torch
print(f'GPU  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — check runtime type!"}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/Research/multi_model/object_detection'
os.makedirs(f'{WORK_DIR}/data/zips', exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
print(f'Work dir: {WORK_DIR}')

# Clone code
%cd /content
!git clone https://github.com/DaniYLab/object_detection.git 2>/dev/null || (cd object_detection && git pull)
%cd /content/object_detection


## 2 — Install dependencies


In [ ]:
# Colab đã có torch CUDA — chỉ cài thêm thư viện khác
!pip install -q gdown Pillow tqdm svgpathtools matplotlib

import torch
print(f'torch : {torch.__version__}')
print(f'cuda  : {torch.version.cuda}')


## 3A — LẦN ĐẦU: Download zip gốc về Drive
> ⏭️ Skip nếu đã có `Drive/.../data/zips/` rồi


In [ ]:
# Trỏ data/ trong repo → Drive (symlink)
import subprocess, os
!rm -f /content/object_detection/data
subprocess.run(['ln', '-sf', f'{WORK_DIR}/data', '/content/object_detection/data'], check=True)

# Download 3 zips vào Drive
!python scripts/data/download_gdrive.py


## 3B — LẦN ĐẦU: Build dataset (toàn bộ trên SSD)
> ⏭️ Skip nếu đã có `Drive/.../data/dataset.tar.gz` rồi


In [ ]:
import os

# Unzip raw data từ Drive vào SSD
!mkdir -p /content/FloorPlanCAD_orig
for s in ["train_set_1", "train_set_2", "test_set"]:
    print(f'Extracting {s}...')
    os.system(f'unzip -q "{WORK_DIR}/data/FloorPlanCAD_original/zips/{s}.zip" -d /content/FloorPlanCAD_orig/')
print('Done extracting!')


In [ ]:
# Build images + metadata on SSD
!python scripts/data/build_dataset.py \
    --original_root /content/FloorPlanCAD_orig \
    --output_root   /content/FloorPlanCAD_data


In [ ]:
# Tar và lưu sang Drive
import subprocess, os
!tar -czf /content/dataset.tar.gz -C /content FloorPlanCAD_data
subprocess.run(['mv', '/content/dataset.tar.gz', f'{WORK_DIR}/data/dataset.tar.gz'], check=True)
SIZE = os.path.getsize(f'{WORK_DIR}/data/dataset.tar.gz') / 1e9
print(f'Saved! Size: {SIZE:.2f} GB')


## 4 — Mỗi session: Load dataset từ Drive
> ✅ Chạy cell này mỗi lần mở session mới


In [ ]:
import os, subprocess

if not os.path.exists('/content/FloorPlanCAD_data'):
    print('Extracting dataset from Drive to SSD...')
    subprocess.run(['tar', '-xzf', f'{WORK_DIR}/data/dataset.tar.gz', '-C', '/content/'], check=True)
    print('Done!')
else:
    print('Dataset already on SSD.')

!rm -f /content/object_detection/data
!mkdir -p /content/object_detection/data
!ln -sf /content/FloorPlanCAD_data /content/object_detection/data/FloorPlanCAD_dataset
!ls /content/object_detection/data/


## 5 — Verify dataset + model


In [ ]:
import sys
sys.path.insert(0, '/content/object_detection')

from src.data.dataset import FloorPlanDataset, CLASS_NAMES, NUM_CLASSES
from src.models.detector import FloorPlanDetector

train_ds = FloorPlanDataset('./data/FloorPlanCAD_dataset', split='train', image_size=512)
val_ds   = FloorPlanDataset('./data/FloorPlanCAD_dataset', split='test',  image_size=512)
print(f'Train : {len(train_ds):,} samples')
print(f'Val   : {len(val_ds):,} samples')
print(f'Classes: {NUM_CLASSES}')

s = train_ds[0]
print(f'Image         : {s["image"].shape}')
print(f'center_heatmap: {s["center_heatmap"].shape}')
print(f'size_map      : {s["size_map"].shape}')
print(f'text          : {s["text"]}')
print(f'class_id      : {s["class_id"]} ({CLASS_NAMES[s["class_id"]]})')

model = FloorPlanDetector(image_size=512, model_dim=256, num_classes=NUM_CLASSES, depth_per_class=2)
n = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\nModel params: {n:.1f}M')


## 6 — Training

| GPU | VRAM | batch_size |
|-----|------|------------|
| T4  | 16GB | 4          |
| L4  | 24GB | 8          |
| A100| 40GB | 16         |


In [ ]:
import os

CKPT_DIR = f'{WORK_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

!python train.py \
    --data_root    ./data/FloorPlanCAD_dataset \
    --ckpt_dir     {CKPT_DIR} \
    --image_size   512 \
    --batch_size   4 \
    --num_workers  2 \
    --model_dim    256 \
    --depth_per_class 2 \
    --epochs       50 \
    --lr           1e-4 \
    --log_interval 50


## 7 — Load checkpoint & Visualize


In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from src.models.detector import FloorPlanDetector
from src.data.dataset import FloorPlanDataset, CLASS_NAMES, NUM_CLASSES

CKPT_PATH = f'{WORK_DIR}/checkpoints/best.pt'
ckpt  = torch.load(CKPT_PATH, map_location='cpu')
model = FloorPlanDetector(image_size=512, model_dim=256, num_classes=NUM_CLASSES, depth_per_class=2)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Epoch {ckpt["epoch"]} | Val loss: {ckpt["val_loss"]:.4f}')

ds = FloorPlanDataset('./data/FloorPlanCAD_dataset', split='test', image_size=512)
sample = ds[0]

# Inference: all 35 classes (no class_ids = run all blocks)
img = sample['image'].unsqueeze(0)  # [1, 3, 512, 512]
with torch.no_grad():
    preds = model(img)  # class_ids=None -> all 35 classes

hm = preds['center_heatmap'][0]  # [35, 64, 64]
print(f'Heatmap shape: {hm.shape}')

# Show top-5 active classes
class_scores = hm.max(dim=-1).values.max(dim=-1).values  # [35]
top5 = class_scores.topk(5)
for score, idx in zip(top5.values, top5.indices):
    print(f'  {CLASS_NAMES[idx]:20s} max={score:.3f}')

# Visualize top class heatmap
best_cid = top5.indices[0].item()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
img_np = (sample['image'].permute(1,2,0).numpy() * 0.5 + 0.5).clip(0, 1)
axes[0].imshow(img_np)
axes[0].set_title('Original')
axes[1].imshow(hm[best_cid].numpy(), cmap='hot')
axes[1].set_title(f'{CLASS_NAMES[best_cid]} heatmap')
plt.tight_layout()
plt.show()
